In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

# ==========================
# LOAD DATASET
# ==========================
data = pd.read_excel("student_marks_10000 (1).xlsx")

# Encode Pass/Fail
le = LabelEncoder()
data["Pass"] = le.fit_transform(data["Result"])

# ==========================
# REGRESSION SETUP
# ==========================
X_reg = data[["Study_Hours","Attendance","Assignment_Score"]]
y_reg = data["Final_Marks"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg,y_reg,test_size=0.2,random_state=42)
reg_model = LinearRegression()
reg_model.fit(X_train_reg,y_train_reg)
reg_pred = reg_model.predict(X_test_reg)

# ==========================
# CLASSIFICATION SETUP
# ==========================
X_cls = data[["Study_Hours","Attendance","Assignment_Score"]]
y_cls = data["Pass"]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X_cls,y_cls,test_size=0.2,random_state=42)
cls_model = LogisticRegression(max_iter=1000)
cls_model.fit(X_train_cls,y_train_cls)
cls_pred = cls_model.predict(X_test_cls)

# ==========================
# DARK THEME
# ==========================
plt.style.use("dark_background")
sns.set_theme(style="darkgrid")

# ==========================
# WIDGETS
# ==========================
hours_slider = widgets.IntSlider(value=6,min=0,max=12,step=1,description="Study Hours")
attendance_slider = widgets.IntSlider(value=85,min=0,max=100,step=5,description="Attendance %")
assignment_slider = widgets.IntSlider(value=8,min=0,max=10,step=1,description="Assignment Score")

predict_button = widgets.Button(description="Predict",button_style="success")
metrics_button = widgets.Button(description="Show Metrics",button_style="info")
reset_button = widgets.Button(description="Reset",button_style="warning")

output = widgets.Output()

# ==========================
# FUNCTIONS
# ==========================
def update_dashboard(change=None):
    with output:
        clear_output(wait=True)

        # New student data
        new_student = pd.DataFrame([[hours_slider.value,attendance_slider.value,assignment_slider.value]],
                                   columns=["Study_Hours","Attendance","Assignment_Score"])

        # --- Regression Prediction ---
        reg_prediction = reg_model.predict(new_student)[0]
        print(f"📊 Predicted Final Marks: {reg_prediction:.2f}")
        print(f"Regression R² Score: {r2_score(y_test_reg,reg_pred):.2f}")
        print(f"Regression MSE: {mean_squared_error(y_test_reg,reg_pred):.2f}")

        # --- Classification Prediction ---
        cls_prediction = cls_model.predict(new_student)[0]
        result = "✅ PASS" if cls_prediction==1 else "❌ FAIL"
        print(f"Classification Accuracy: {accuracy_score(y_test_cls,cls_pred):.2f}")
        print("Prediction:", result)

        # --- Visuals ---
        fig, ax = plt.subplots(1,3,figsize=(15,4))

        # Regression scatter
        ax[0].scatter(y_test_reg,reg_pred,color="cyan",label="Predicted")
        ax[0].plot([y_test_reg.min(),y_test_reg.max()],
                   [y_test_reg.min(),y_test_reg.max()],
                   color="magenta",linestyle="--")
        ax[0].set_title("Actual vs Predicted Marks")
        ax[0].set_xlabel("Actual")
        ax[0].set_ylabel("Predicted")

        # Confusion Matrix
        cm = confusion_matrix(y_test_cls,cls_pred)
        sns.heatmap(cm,annot=True,fmt="d",cmap="magma",
                    xticklabels=["Fail","Pass"],yticklabels=["Fail","Pass"],ax=ax[1])
        ax[1].set_title("Pass/Fail Confusion Matrix")

        # Pass vs Fail Distribution
        data["Result"].value_counts().plot(kind="bar",color=["crimson","lime"],ax=ax[2])
        ax[2].set_title("Pass vs Fail Distribution")
        ax[2].set_xlabel("Result")
        ax[2].set_ylabel("Count")

        plt.show()

def show_metrics(change=None):
    with output:
        clear_output(wait=True)
        print("📈 Regression Metrics")
        print(f"R² Score: {r2_score(y_test_reg,reg_pred):.2f}")
        print(f"MSE: {mean_squared_error(y_test_reg,reg_pred):.2f}")
        print("\n📊 Classification Metrics")
        print(classification_report(y_test_cls,cls_pred))

def reset_dashboard(change=None):
    hours_slider.value=6
    attendance_slider.value=85
    assignment_slider.value=8
    update_dashboard()

predict_button.on_click(update_dashboard)
metrics_button.on_click(show_metrics)
reset_button.on_click(reset_dashboard)

# ==========================
# DASHBOARD LAYOUT
# ==========================
dashboard = widgets.VBox([
    widgets.HBox([hours_slider,attendance_slider,assignment_slider]),
    widgets.HBox([predict_button,metrics_button,reset_button]),
    output
])
display(dashboard)

# Initial run
update_dashboard()
